<a href="https://colab.research.google.com/github/mmanana/pypacity/blob/master/weather_info_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Utilización de datos meteorológicos disponibles online para el cálculo de la capacidad dinámica de conductores

## Autor: Mario Mañana Canteli
## Última actualización: 23/9/2022

Identificar versión del compilador y ubicación.

In [1]:
import sys
print(sys.version)
print(sys.executable)

3.7.14 (default, Sep  8 2022, 00:06:44) 
[GCC 7.5.0]
/usr/bin/python3


Instalar paquetes que no están disponibles por defecto en las VM.

IMPORTANTE: Ejecutar un única vez por sesión para reducir tiempo de ejecución.

In [2]:
!pip install pyodbc pyzipper yagmail

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 472 kB 8.0 MB/s 
     |████████████████████████████████| 67 kB 6.1 MB/s 
     |████████████████████████████████| 2.3 MB 42.5 MB/s 
     |████████████████████████████████| 399 kB 51.4 MB/s 


Cargar paquetes básicos.

In [3]:
import os 
import math
import numpy as np
import pandas as pd
import scipy.integrate as integrate
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge, Polygon, Ellipse
from matplotlib.collections import PatchCollection
from IPython.display import Image, HTML, SVG, YouTubeVideo
from datetime import date, timedelta, datetime
import requests
import pyodbc
import configparser
import pyzipper
import yagmail
import http.client

Definir si el fichero notebook se ejecuta en local o en Google Colab

Google Collaborate => IsColab = 1

Máquina local => IsColab = 0

In [4]:
IsColab = 1 # 1 .- Google collaborate; 0 .- Local

if IsColab == 1:
  from google.colab import drive
  drive.mount('/content/drive')
  path = "/content/drive/MyDrive/Colab Notebooks/DTR_Iberdrola/"

Mounted at /content/drive


Parámetros de configuración

In [5]:
if IsColab == 1:
  file = path + "config.ini"
else:
  file = "./fconfig.ini"


configP = configparser.ConfigParser()
configP.read(file)

['/content/drive/MyDrive/Colab Notebooks/DTR_Iberdrola/config.ini']

# **OpenWeatherMap**

---



Fuente: OpenWeatherMap 
Método de acceso: API key
Endpoint: http://api.openweathermap.org/

In [ ]:
key = configP['GENERAL']['KEY']

Subscription to Free OpenWeatherMap!

API key:
- Your API key is 02a6e99fcf3fbad7d84ea4e8e4f65333
- Within the next couple of hours, it will be activated and ready to use
- You can later create more API keys on your account page
- Please, always use your API key in each API call

Endpoint:
- Please, use the endpoint api.openweathermap.org for your API calls
- Example of API call:
api.openweathermap.org/data/2.5/weather?q=London,uk&APPID=02a6e99fcf3fbad7d84ea4e8e4f65333

Useful links:
- API documentation https://openweathermap.org/api
- Details of your plan https://openweathermap.org/price
- Please, note that 16-days daily forecast and History API are not available for Free subscribers


Blog
Support center & FAQ
Contact us info@openweathermap.org. 


In [ ]:
# url = "https://api.openweathermap.org/data/2.5/weather?lat=44.34&lon=10.99&appid={bdb4917d51df827b1ed9bd7aa03b1f8f}"

In [ ]:
3 url = "https://community-open-weather-map.p.rapidapi.com/weather"
# querystring = {"q":"London%2Cuk"}
# headers = {
    #'x-rapidapi-host': "community-open-weather-map.p.rapidapi.com",
    # 'x-rapidapi-key': "[02a6e99fcf3fbad7d84ea4e8e4f65333]"
    #}
# response = requests.request("GET", url, headers=headers, params=querystring)
# print(response.text)

{"message":"You are not subscribed to this API."}


In [ ]:
# conn = http.client.HTTPSConnection("community-open-weather-map.p.rapidapi.com")
#headers = {
 #   'x-rapidapi-host': "community-open-weather-map.p.rapidapi.com",
 #   'x-rapidapi-key': "[02a6e99fcf3fbad7d84ea4e8e4f65333]"
 #   }
# conn.request("GET", "/weather?q=London%252Cuk", headers=headers)
# res = conn.getresponse()
# data = res.read()
# print(data.decode("utf-8"))

{"message":"You are not subscribed to this API."}


# **Open Meteo**

---

https://open-meteo.com/

Ofrece datos en abierto sin necesidad de generar keys

In [ ]:
url = "https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&hourly=temperature_2m"
response = requests.get(url)

In [ ]:
response.json()

{'latitude': 52.52,
 'longitude': 13.419998,
 'generationtime_ms': 0.4069805145263672,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 38.0,
 'current_weather': {'temperature': 11.7,
  'windspeed': 2.82,
  'winddirection': 174.0,
  'weathercode': 61.0,
  'time': '2022-09-26T23:00'},
 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C'},
 'hourly': {'time': ['2022-09-26T00:00',
   '2022-09-26T01:00',
   '2022-09-26T02:00',
   '2022-09-26T03:00',
   '2022-09-26T04:00',
   '2022-09-26T05:00',
   '2022-09-26T06:00',
   '2022-09-26T07:00',
   '2022-09-26T08:00',
   '2022-09-26T09:00',
   '2022-09-26T10:00',
   '2022-09-26T11:00',
   '2022-09-26T12:00',
   '2022-09-26T13:00',
   '2022-09-26T14:00',
   '2022-09-26T15:00',
   '2022-09-26T16:00',
   '2022-09-26T17:00',
   '2022-09-26T18:00',
   '2022-09-26T19:00',
   '2022-09-26T20:00',
   '2022-09-26T21:00',
   '2022-09-26T22:00',
   '2022-09-26T23:00'],
  'temperature_2m': [11.7,
   11.4,
  

In [ ]:
json_response = response.json()
repository = json_response['elevation'][0]
#print(f'Repository name: {repository["current_weather"]}')  # Python 3.6+

TypeError: ignored